# Rent model sandbox

Loads current `housing_listings` from Supabase. The model constructors live in the cells below — change them and re-run the cell:

- in-sample bake-off (OLS / random forest / XGBoost / spatial Durbin)
- spatial block CV
- property-grouped out-of-fold fair rent, plus CMA

Set `DB_URI` in `airflow/.env` (or your environment). This notebook does not write back to the DB unless you uncomment the update cell.

In [21]:
from pathlib import Path
import sys
import os

import pandas as pd
import numpy as np

# Resolve model package root whether you launch from repo root or notebooks/
NOTEBOOK_DIR = Path.cwd().resolve()
CANDIDATES = [
    NOTEBOOK_DIR / "model",
    NOTEBOOK_DIR.parent / "model",
    NOTEBOOK_DIR.parent.parent / "model",
    Path("/opt/airflow/model"),
]
MODEL_PATH = next((p for p in CANDIDATES if (p / "ml" / "model_training.py").exists()), None)
if MODEL_PATH is None:
    raise FileNotFoundError(f"Could not find airflow/model from {NOTEBOOK_DIR}")
MODEL_PATH = str(MODEL_PATH.resolve())
if MODEL_PATH not in sys.path:
    sys.path.insert(0, MODEL_PATH)
print("MODEL_PATH:", MODEL_PATH)

# Load DB_URI from airflow/.env if present
ENV_CANDIDATES = [
    Path(MODEL_PATH).parent / ".env",
    Path(MODEL_PATH) / ".env",
    NOTEBOOK_DIR / ".env",
]
try:
    from dotenv import load_dotenv
    for env_path in ENV_CANDIDATES:
        if env_path.exists():
            load_dotenv(env_path, override=False)
            print("Loaded env:", env_path)
            break
except ImportError:
    print("python-dotenv not installed; relying on existing environment vars")

assert os.getenv("DB_URI"), "Set DB_URI (e.g. in airflow/.env)"

MODEL_PATH: /Users/asm/Documents/GitHub/IthacaHousing/airflow/model
Loaded env: /Users/asm/Documents/GitHub/IthacaHousing/airflow/.env


In [2]:
import ml.model_training as model_training
import ml.data_preprocessing as data_preprocessing
import ml.comparative_market_analysis as comparative_market_analysis
import ml.rerun_rent_prediction as rerun_rent_prediction

## 1) Load + prepare from Supabase
Uses `fetch_listings_for_retrain` / `prepare_features_from_db` (no scrape).

In [22]:
raw = rerun_rent_prediction.fetch_listings_for_retrain()
apartments = rerun_rent_prediction.prepare_features_from_db(raw)

print(apartments.shape)
apartments[["ListingId", "PropertyId", "rent_per_person", "latitude", "longitude"]].head()

📊 Loaded 1037 listings from housing_listings (43 columns)
Calculating rent adjustments using extracted data...
Adding property features from ownership data...
✅ Added sqft_per_sale_price feature for 0 valid records
✅ Added YR_BUILT_ENCODED feature
🧹 Trainable rows after rent/coord filter: 902/1037 (185 properties)
(902, 72)


,ListingId,PropertyId,rent_per_person,latitude,longitude
0,118377-30764259,118377,925.0,42.440941,-76.491440
1,242367-30764129,242367,1649.0,42.426498,-76.448929
2,7734-17720,7734,1635.0,42.440170,-76.474800
3,7734-17721,7734,1880.0,42.440170,-76.474800
4,118348-30639850,118348,850.0,42.435658,-76.493378


In [23]:
pd.set_option('display.max_columns', None)
apartments[apartments["renttype"] != "Price Per Unit"]

,listingid,listingaddress,listingcity,listingzip,latitude,longitude,rentamount,renttype,rent_per_person,num_people,total_rent_amount,bedrooms,bathrooms,available_bedrooms,available_bathrooms,pets,amenities,amenities_score,transit_score,valid_certificate_of_compliance,year_built,sale_price,assessment_sqft,neighborhood,owner_name,lengthavailable,walk_time_urishall,walk_time_agriculturequad,walk_time_artsquad,walk_time_engineeringquad,bike_time_urishall,bike_time_agriculturequad,bike_time_artsquad,bike_time_engineeringquad,drive_time_urishall,drive_time_agriculturequad,drive_time_artsquad,drive_time_engineeringquad,nearest_stop_name,walk_time_to_nearest_stop,transit_time_to_ag_quad,transit_time_to_arts_quad,transit_time_to_eng_quad,ListingId,PropertyId,Bedrooms,Bathrooms,Pets,Amenities,LengthAvailable,Electricity Included,Heat Included,Internet Available,Internet Included,Kitchen,Laundry Facilities,Air Conditioning,Furnished,Near Bus Route,Wheelchair Accessible,Off-Street Parking Included,Electronic Payments Accepted,Water Included,Off-Street Parking Available,Permitted Street Parking Available,bedroom_bathroom_ratio,available_bedrooms_to_total_bedrooms_ratio,combined_bedrooms_bathrooms,sqft_per_sale_price,YR_BUILT_ENCODED,GmapLatitude,GmapLongitude
18,7751-17773,614 STEWART AVE,Ithaca,14850.0,42.446770,-76.490471,850.0,Price per Person,850.0,1.0,850.0,1.0,2.0,1.0,1.0,0,"[""Dishwasher"", ""Fireplace"", ""Furnished"", ""Hard...",83.33,51.01,0,NaN,NaN,None,None,Not Found,None,9.03,14.29,7.96,10.38,4.61,7.14,3.95,3.82,2.78,4.26,2.78,2.14,University @ Cornell Ave,2.743689,21.0,14.0,16.0,7751-17773,7751,1.0,2.0,0,"[Dishwasher, Fireplace, Furnished, Hardwood Fl...",None,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0.75,1.0,2.5,NaN,0,42.446770,-76.490471
19,7751-17774,614 STEWART AVE,Ithaca,14850.0,42.446770,-76.490471,970.0,Price per Person,970.0,2.0,1940.0,2.0,1.0,2.0,1.0,0,"[""Dishwasher"", ""Fireplace"", ""Furnished"", ""Hard...",83.33,51.01,0,NaN,NaN,None,None,Not Found,None,9.03,14.29,7.96,10.38,4.61,7.14,3.95,3.82,2.78,4.26,2.78,2.14,University @ Cornell Ave,2.743689,21.0,14.0,16.0,7751-17774,7751,2.0,1.0,0,"[Dishwasher, Fireplace, Furnished, Hardwood Fl...",None,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,3.00,1.0,4.0,NaN,0,42.446770,-76.490471
20,7751-17775,614 STEWART AVE,Ithaca,14850.0,42.446770,-76.490471,900.0,Price per Person,900.0,1.0,900.0,1.0,2.0,1.0,1.0,0,"[""Dishwasher"", ""Fireplace"", ""Furnished"", ""Hard...",83.33,51.01,0,NaN,NaN,None,None,Not Found,None,9.03,14.29,7.96,10.38,4.61,7.14,3.95,3.82,2.78,4.26,2.78,2.14,University @ Cornell Ave,2.743689,21.0,14.0,16.0,7751-17775,7751,1.0,2.0,0,"[Dishwasher, Fireplace, Furnished, Hardwood Fl...",None,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0.75,1.0,2.5,NaN,0,42.446770,-76.490471
21,7751-30697460,614 STEWART AVE,Ithaca,14850.0,42.446770,-76.490471,970.0,Price per Person,970.0,5.0,4850.0,5.0,2.0,5.0,1.0,0,"[""Dishwasher"", ""Fireplace"", ""Furnished"", ""Hard...",83.33,51.01,0,NaN,NaN,None,None,Not Found,None,9.03,14.29,7.96,10.38,4.61,7.14,3.95,3.82,2.78,4.26,2.78,2.14,University @ Cornell Ave,2.743689,21.0,14.0,16.0,7751-30697460,7751,5.0,2.0,0,"[Dishwasher, Fireplace, Furnished, Hardwood Fl...",None,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,3.75,1.0,8.5,NaN,0,42.446770,-76.490471
22,7751-30697464,614 STEWART AVE,Ithaca,14850.0,42.446770,-76.490471,970.0,Price per Person,970.0,6.0,5820.0,6.0,2.0,6.0,1.0,0,"[""Dishwasher"", ""Fireplace"", ""Furnished"", ""Hard...",83.33,51.01,0,NaN,NaN,None,None,Not Found,None,9.03,14.29,7.96,10.38,4.61,7.14,3.95,3.82,2.78,4.26,2.78,2.14,University @ Cornell Ave,2.743689,21.0,14.0,16.0,7751-30697464,7751,6.0,2.0,0,"[Dishwasher, Fireplace, Furnished, Hardwood Fl...",None,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,4.50,1.0,10.0,NaN,0,42.446770,-76.490471
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,

In [28]:
p25 = apartments["rent_per_person"].quantile(0.25)
p75 = apartments["rent_per_person"].quantile(0.75)
iqr = p75-p25
median = apartments["rent_per_person"].median()
iqr_factor = 3

iqr_test = (apartments["rent_per_person"] > median+iqr*iqr_factor) | (apartments["rent_per_person"] < median-iqr*iqr_factor)
apartments[iqr_test]

,listingid,listingaddress,listingcity,listingzip,latitude,longitude,rentamount,renttype,rent_per_person,num_people,total_rent_amount,bedrooms,bathrooms,available_bedrooms,available_bathrooms,pets,amenities,amenities_score,transit_score,valid_certificate_of_compliance,year_built,sale_price,assessment_sqft,neighborhood,owner_name,lengthavailable,walk_time_urishall,walk_time_agriculturequad,walk_time_artsquad,walk_time_engineeringquad,bike_time_urishall,bike_time_agriculturequad,bike_time_artsquad,bike_time_engineeringquad,drive_time_urishall,drive_time_agriculturequad,drive_time_artsquad,drive_time_engineeringquad,nearest_stop_name,walk_time_to_nearest_stop,transit_time_to_ag_quad,transit_time_to_arts_quad,transit_time_to_eng_quad,ListingId,PropertyId,Bedrooms,Bathrooms,Pets,Amenities,LengthAvailable,Electricity Included,Heat Included,Internet Available,Internet Included,Kitchen,Laundry Facilities,Air Conditioning,Furnished,Near Bus Route,Wheelchair Accessible,Off-Street Parking Included,Electronic Payments Accepted,Water Included,Off-Street Parking Available,Permitted Street Parking Available,bedroom_bathroom_ratio,available_bedrooms_to_total_bedrooms_ratio,combined_bedrooms_bathrooms,sqft_per_sale_price,YR_BUILT_ENCODED,GmapLatitude,GmapLongitude


## 2) Build X / y

In [29]:
X, y = model_training.define_X_Y_variables(apartments)
X, y = data_preprocessing.clean_up_x_y(X, y)
X = data_preprocessing.median_mode_imputation(X)
y_log = data_preprocessing.log_transform_prices(y)

X = X.reset_index(drop=True)
y_log = y_log.reset_index(drop=True) if hasattr(y_log, "reset_index") else pd.Series(y_log).reset_index(drop=True)
apartments = apartments.reset_index(drop=True)

print("Features:", list(X.columns))
print("n =", len(X), "| y (log) mean =", float(np.mean(y_log)))
X.head()

📊 Using 14 features for model training
📋 Feature columns: ['Pets', 'combined_bedrooms_bathrooms', 'walk_time_urishall', 'transit_score', 'valid_certificate_of_compliance', 'YR_BUILT_ENCODED', 'Electricity Included', 'Heat Included', 'Water Included', 'Internet Included', 'Laundry Facilities', 'Air Conditioning', 'Furnished', 'valid_certificate_of_compliance']
Features: ['Pets', 'combined_bedrooms_bathrooms', 'walk_time_urishall', 'transit_score', 'valid_certificate_of_compliance', 'YR_BUILT_ENCODED', 'Electricity Included', 'Heat Included', 'Water Included', 'Internet Included', 'Laundry Facilities', 'Air Conditioning', 'Furnished']
n = 902 | y (log) mean = 6.974154904244657


,Pets,combined_bedrooms_bathrooms,walk_time_urishall,transit_score,valid_certificate_of_compliance,YR_BUILT_ENCODED,Electricity Included,Heat Included,Water Included,Internet Included,Laundry Facilities,Air Conditioning,Furnished
0,0,4.0,18.47,100.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0,2.5,39.03,0.00,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0,2.5,15.19,56.09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0,2.5,15.19,56.09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0,5.5,30.74,51.83,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3) Classic model bake-off

Linear regression, random forest, and XGBoost fit on spatial lags of the features. Spatial Durbin fits on `X` directly and builds its own neighbor weights.

Edit the constructors at the top of the next cell and re-run it. Metrics are in-sample, so use them to compare models, not as honest error. An R² above 0.96 is flagged and left out of the champion pick.

In [ ]:
import statsmodels.api as sm
from libpysal.weights import KNN
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from spreg import ML_Lag
from xgboost import XGBRegressor

# --- edit these, then re-run the cell ---
RF_PARAMS = dict(n_estimators=200, max_depth=12, random_state=42)
XGB_PARAMS = dict(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
DURBIN_K = 12
OVERFIT_R2 = 0.96


def make_random_forest():
    return RandomForestRegressor(**RF_PARAMS)


def make_xgboost():
    return XGBRegressor(**XGB_PARAMS)


def _clean_xy(X_in, y_in):
    Xc = X_in.copy()
    for col in Xc.columns:
        Xc[col] = pd.to_numeric(Xc[col], errors="coerce")
    Xc = Xc.replace([np.inf, -np.inf], np.nan).fillna(Xc.median(numeric_only=True)).fillna(0)
    Xc = Xc.astype("float64")
    yc = np.nan_to_num(np.asarray(y_in, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    return Xc, yc


def _attach_prediction(df, y_pred):
    out = df.copy()
    out["PredictedRent"] = np.exp(np.asarray(y_pred, dtype=float))
    out["DifferenceinFairValue"] = out["rent_per_person"] - out["PredictedRent"]
    return out


def fit_linear(X_in, y_in, df):
    Xc, yc = _clean_xy(X_in, y_in)
    X_const = sm.add_constant(Xc, has_constant="add")
    y_pred = np.asarray(sm.OLS(yc, X_const).fit().predict(X_const), dtype=float)
    return _attach_prediction(df, y_pred), y_pred


def fit_sklearn(make_model, X_in, y_in, df):
    Xc, yc = _clean_xy(X_in, y_in)
    model = make_model()
    model.fit(Xc, yc)
    y_pred = np.asarray(model.predict(Xc), dtype=float)
    return _attach_prediction(df, y_pred), y_pred


def fit_durbin(X_in, y_in, df):
    Xc, yc = _clean_xy(X_in, y_in)
    weights = KNN.from_array(df[["latitude", "longitude"]].to_numpy(), k=DURBIN_K)
    weights.transform = "R"
    try:
        sdm = ML_Lag(
            yc,
            Xc,
            w=weights,
            name_y="Log rent_per_person",
            name_x=Xc.columns.tolist(),
            slx_lags=1,
        )
        y_pred = np.asarray(sdm.predy).ravel()
    except (np.linalg.LinAlgError, ValueError) as exc:
        print(f"Spatial Durbin failed ({exc}); falling back to OLS")
        X_const = sm.add_constant(Xc, has_constant="add")
        y_pred = np.asarray(sm.OLS(yc, X_const).fit().predict(X_const), dtype=float)
    return _attach_prediction(df, y_pred), y_pred


X_spatial_eval = model_training.get_spatial_coefficients(X, apartments)
X_eval = X
y_eval = np.asarray(y_log, dtype=float)

models = {
    "LinearRegression (Spatial)": lambda: fit_linear(X_spatial_eval, y_eval, apartments),
    "RandomForest (Spatial)": lambda: fit_sklearn(make_random_forest, X_spatial_eval, y_eval, apartments),
    "XGBoost (Spatial)": lambda: fit_sklearn(make_xgboost, X_spatial_eval, y_eval, apartments),
    "Spatial Durbin": lambda: fit_durbin(X, y_eval, apartments),
}

rows = {}
fitted = {}
for name, runner in models.items():
    try:
        apts_out, y_pred = runner()
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        r2 = float(r2_score(y_eval, y_pred))
        mape = float(mean_absolute_percentage_error(y_eval, y_pred))
        rows[name] = {"RMSE": rmse, "R2": r2, "MAPE": mape, "flagged": r2 > OVERFIT_R2}
        fitted[name] = apts_out
        print(f"{name}: R2={r2:.4f} RMSE={rmse:.4f} MAPE={mape:.4f} flagged={r2 > OVERFIT_R2}")
    except Exception as exc:
        print(f"Skipping {name}: {exc}")

results_classic = pd.DataFrame(rows).T
if results_classic.empty:
    raise RuntimeError("All models failed to train")

eligible = results_classic.loc[~results_classic["flagged"].astype(bool)].copy()
if eligible.empty:
    raise RuntimeError("All models were flagged as overfit")

eligible["RMSE_norm"] = 1 - (eligible["RMSE"] - eligible["RMSE"].min()) / (
    eligible["RMSE"].max() - eligible["RMSE"].min() + 1e-9
)
eligible["Score"] = 0.5 * eligible["R2"] + 0.5 * eligible["RMSE_norm"]
best_name = eligible["Score"].idxmax()
print("Best:", best_name)

apts_classic = fitted[best_name]
results_classic = results_classic.join(eligible[["RMSE_norm", "Score"]], how="left")
results_classic.sort_values("Score", ascending=False)


## 4) Spatial block CV

Honest holdout by spectral spatial cluster. Edit `make_block_model` and `make_block_baseline` at the top of the next cell. That cell also defines `add_fold_spatial_features`, which the out-of-fold cell reuses.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# --- edit these, then re-run the cell ---
H3_RESES = (6, 7)
K_KNN = 10
N_CLUSTERS = 6


def make_block_model():
    return XGBRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
    )


def make_block_baseline():
    """Train-only model whose residuals become the KNN residual lag. Not the scored model."""
    return XGBRegressor(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
    )


def _as_numeric(frame):
    out = frame.copy()
    for col in out.columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out.replace([np.inf, -np.inf], np.nan, inplace=True)
    out.fillna(0, inplace=True)
    return out


def add_fold_spatial_features(X_train, X_test, y_train, h3_reses, k_knn, make_baseline):
    """H3 target encoding and KNN lags fit on the training fold only."""
    X_train = X_train.copy()
    X_test = X_test.copy()
    y_train = np.asarray(y_train, dtype=float)
    y_train_series = pd.Series(y_train, index=X_train.index)
    global_mean = float(y_train_series.mean())

    for res in h3_reses:
        h3_col = f"h3_{res}"
        te_col = f"h3_{res}_te"
        if h3_col not in X_train.columns:
            continue
        cell_means = y_train_series.groupby(X_train[h3_col]).mean()
        X_train[te_col] = X_train[h3_col].map(cell_means)
        X_test[te_col] = X_test[h3_col].map(cell_means).fillna(global_mean)

    drop_cols = [f"h3_{res}" for res in h3_reses if f"h3_{res}" in X_train.columns]
    X_train = X_train.drop(columns=drop_cols, errors="ignore")
    X_test = X_test.drop(columns=drop_cols, errors="ignore")
    X_train = _as_numeric(X_train)
    X_test = _as_numeric(X_test)

    X_train_base = X_train.drop(columns=["latitude", "longitude"], errors="ignore")
    baseline = make_baseline()
    baseline.fit(X_train_base, y_train)
    train_resid = y_train - baseline.predict(X_train_base)

    rent_lag_tr, rent_lag_te = model_training.knn_distance_weighted_lag_train_test(
        X_train, X_test, y_train, k=k_knn
    )
    X_train[f"knn_rent_dw_{k_knn}"] = rent_lag_tr
    X_test[f"knn_rent_dw_{k_knn}"] = rent_lag_te

    resid_lag_tr, resid_lag_te = model_training.knn_distance_weighted_lag_train_test(
        X_train, X_test, train_resid, k=k_knn
    )
    X_train[f"knn_resid_dw_{k_knn}"] = resid_lag_tr
    X_test[f"knn_resid_dw_{k_knn}"] = resid_lag_te

    if "YR_BUILT_ENCODED" in X_train.columns:
        year_lag_tr, year_lag_te = model_training.knn_distance_weighted_lag_train_test(
            X_train, X_test, X_train["YR_BUILT_ENCODED"].to_numpy(), k=k_knn
        )
        X_train[f"knn_yearbuilt_dw_{k_knn}"] = year_lag_tr
        X_test[f"knn_yearbuilt_dw_{k_knn}"] = year_lag_te

    model_drop = ["latitude", "longitude", "residual"]
    return (
        _as_numeric(X_train.drop(columns=model_drop, errors="ignore")),
        _as_numeric(X_test.drop(columns=model_drop, errors="ignore")),
    )


X_coords = X.copy()
X_coords["latitude"] = apartments["latitude"].values
X_coords["longitude"] = apartments["longitude"].values
X_coords = model_training.add_h3_cells(X_coords, h3_reses=H3_RESES)
X_coords = model_training.spatial_spectral_clustering(X_coords, n_clusters=N_CLUSTERS)

x_cols = [
    c for c in X_coords.columns
    if c not in ("spatial_cluster", "latitude", "longitude")
]
y_series = y_log if isinstance(y_log, pd.Series) else pd.Series(y_log, index=X_coords.index)

metrics = []
feature_importances = []
for blk in X_coords["spatial_cluster"].unique():
    test_mask = X_coords["spatial_cluster"] == blk
    train_mask = ~test_mask
    cols = [c for c in list(x_cols) + ["latitude", "longitude"] if c in X_coords.columns]
    X_train = X_coords.loc[train_mask, cols].copy()
    X_test = X_coords.loc[test_mask, cols].copy()
    y_train = y_series.loc[train_mask].to_numpy(dtype=float)
    y_test = y_series.loc[test_mask].to_numpy(dtype=float)

    X_train_model, X_test_model = add_fold_spatial_features(
        X_train, X_test, y_train, H3_RESES, K_KNN, make_block_baseline
    )
    model = make_block_model()
    model.fit(X_train_model, y_train)
    y_pred = np.nan_to_num(model.predict(X_test_model), nan=0.0, posinf=0.0, neginf=0.0)

    row = {
        "block": blk,
        "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
        "mae": float(mean_absolute_error(y_test, y_pred)),
        "r2": float(r2_score(y_test, y_pred)),
        "bias": float(np.mean(y_pred - y_test)),
        "block_size": int(len(y_test)),
    }
    metrics.append(row)
    if hasattr(model, "feature_importances_"):
        feature_importances.append(pd.Series(model.feature_importances_, index=X_train_model.columns))
    print(
        f"Block {blk}: RMSE={row['rmse']:.4f}, R²={row['r2']:.4f}, "
        f"Bias={row['bias']:.4f}, n={row['block_size']}"
    )

metrics_df = pd.DataFrame(metrics)
avg = metrics_df.mean(numeric_only=True).to_dict()
print(
    f"\nMean RMSE: {avg['rmse']:.4f}, Mean R²: {avg['r2']:.4f}, Mean Bias: {avg['bias']:.4f}"
)
if feature_importances:
    fi_mean = pd.DataFrame(feature_importances).mean().sort_values(ascending=False)
    print("\nAverage feature importance")
    display(fi_mean)
display(metrics_df)
avg


## 5) Property-grouped OOF fair rent (+ CMA)

Edit `make_oof_model` and `make_oof_baseline` in the next cell. Every unit from a building stays in the same fold. `fair_rent` is what production stores as `predictedrent`.

Run section 4 first so the H3 columns and `add_fold_spatial_features` exist.

In [ ]:
from sklearn.model_selection import GroupKFold
from xgboost import XGBRegressor

# --- edit these, then re-run the cell ---
N_SPLITS = 5
CMA_NEIGHBORS = 4


def make_oof_model():
    return XGBRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
    )


def make_oof_baseline():
    """Train-only model whose residuals become the KNN residual lag. Not the scored model."""
    return XGBRegressor(
        n_estimators=50,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
    )


apts_oof = apartments.copy()
apts_oof["spatial_cluster"] = X_coords["spatial_cluster"].values
for res in H3_RESES:
    col = f"h3_{res}"
    if col in X_coords.columns:
        apts_oof[col] = X_coords[col].values

y_series = y_log if isinstance(y_log, pd.Series) else pd.Series(y_log, index=apts_oof.index)
y_array = y_series.to_numpy(dtype=float)
groups = model_training.group_by_property_df(apts_oof)
n_splits_eff = min(N_SPLITS, int(groups.nunique()))
if n_splits_eff < 2:
    raise ValueError(f"Need at least 2 properties for grouped OOF, found {groups.nunique()}")

fair_pred = pd.Series(index=apts_oof.index, dtype=float)
gkf = GroupKFold(n_splits=n_splits_eff)
print(
    f"Fair-rent OOF: GroupKFold by property "
    f"({groups.nunique()} properties, {n_splits_eff} folds, {len(apts_oof)} rows)"
)

cols = [c for c in list(x_cols) + ["latitude", "longitude"] if c in apts_oof.columns]
for fold_id, (train_pos, test_pos) in enumerate(gkf.split(apts_oof, y_array, groups)):
    train_idx = apts_oof.index[train_pos]
    test_idx = apts_oof.index[test_pos]
    print(
        f"Fold {fold_id}: train {groups.iloc[train_pos].nunique()} properties / {len(train_pos)} rows, "
        f"test {groups.iloc[test_pos].nunique()} properties / {len(test_pos)} rows"
    )
    X_train = apts_oof.loc[train_idx, cols].copy()
    X_test = apts_oof.loc[test_idx, cols].copy()
    y_train = y_array[train_pos]

    X_train_model, X_test_model = add_fold_spatial_features(
        X_train, X_test, y_train, H3_RESES, K_KNN, make_oof_baseline
    )
    model = make_oof_model()
    model.fit(X_train_model, y_train)
    fair_pred.loc[test_idx] = model.predict(X_test_model)

apts_oof["log_rent"] = y_series
apts_oof["log_fair_rent"] = fair_pred
apts_oof["actual_rent"] = np.exp(apts_oof["log_rent"])
apts_oof["fair_rent"] = np.exp(apts_oof["log_fair_rent"])
apts_oof["mispricing"] = apts_oof["actual_rent"] - apts_oof["fair_rent"]
apts_oof["pct_mispricing"] = apts_oof["mispricing"] / apts_oof["fair_rent"]
apts_oof["mispricing_pct"] = apts_oof["mispricing"] / apts_oof["actual_rent"]

apts_oof["predictedrent"] = apts_oof["fair_rent"]
apts_oof["differenceinfairvalue"] = apts_oof["rent_per_person"] - apts_oof["predictedrent"]

X_cma = comparative_market_analysis.define_X_for_cma(apts_oof)
apts_oof = comparative_market_analysis.perform_cma(X_cma, apts_oof, n_neighbors=CMA_NEIGHBORS)

apts_oof[
    ["ListingId", "rent_per_person", "fair_rent", "predicted_rent_cma", "differenceinfairvalue"]
].describe()


In [8]:
# Quick OOF error (log space vs $ space)
err = apts_oof["rent_per_person"] - apts_oof["fair_rent"]
{
    "mae_$": float(err.abs().mean()),
    "rmse_$": float(np.sqrt((err ** 2).mean())),
    "median_abs_$": float(err.abs().median()),
    "corr": float(apts_oof["rent_per_person"].corr(apts_oof["fair_rent"])),
}

{'mae_$': 460.180739848045,
 'rmse_$': 599.6593121362063,
 'median_abs_$': 374.7347427921143,
 'corr': 0.5326873859579603}

## 6) Optional: write predictions back to Supabase
Uncomment to UPDATE `predictedrent` / CMA / `differenceinfairvalue` only.

In [9]:
# rerun_rent_prediction.update_predictions_in_db(apts_oof)